# 27. Verificações da revisão científica do artigo

Sensibilidades solicitadas na revisão de 24/10/2026. Não alteram o pareamento básico nem os resultados dos NB23–26. As condições cadastrais são evidência observável, não rótulos de erro verdadeiro. O contraste por nível 1 é condicionado aos mesmos quintis e pesos do contraste básico, para conservar a população de padronização. A conciliação com designadores preservados é uma restrição exploratória, não uma política validada.


In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely
import re
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0,str(ROOT))
from src.metrics import chave_identidade, separar_logradouro_numero, conciliar_logradouros
P=ROOT/'data/processed'; T=ROOT/'outputs/referencia_completa/tables'
C=gpd.read_parquet(P/'cnefe_bh.parquet')
B=gpd.read_parquet(P/'bhmap_bh.parquet', columns=['IDEND', 'SIGLA_TIPO', 'NOME_LOGRA', 'NUMERO_IMO', 'NOME_BAIRR', 'NOME_REGIO', 'CEP', 'geometry'])
lc0,nc=chave_identidade(*separar_logradouro_numero(C.LOGRAD_NUM)); lc0.index=nc.index=C.index
lb,nb=chave_identidade(B.SIGLA_TIPO.fillna('')+' '+B.NOME_LOGRA.fillna(''),B.NUMERO_IMO); lb.index=nb.index=B.index
mapa=conciliar_logradouros(lc0.unique(),lb.unique(),limiar=90,margem=2)
lc=lc0.map(lambda s:mapa.get(s,s))
kc=(lc.astype('string')+'|'+nc.astype('string')).where(nc.notna())
kb=(lb.astype('string')+'|'+nb.astype('string')).where(nb.notna())
apto=pd.to_numeric(C.COD_TIPO_ESPECI,errors='coerce').eq(103).fillna(False)
estab=pd.to_numeric(C.COD_ESPECIE,errors='coerce').isin([3,4,5,6,8])
cauda=pd.read_parquet(T/'nb23_cauda_adjudicada.parquet')
assert len(mapa)==1335 and len(cauda)==29775


## 1. Designadores nos nomes conciliados

Sinalizamos mudança dos conjuntos de tokens inteiramente numéricos e letras isoladas ou números romanos ao final do nome. A lista é para inspeção: diferenças de designador podem ser erros, renomeações ou variantes legítimas. A sensibilidade rejeita essas substituições e mede apenas a disponibilidade de chave, sem validar identidade nem recalcular coordenadas.

Designadores escritos por extenso não são reconhecidos automaticamente por esta regra. A lista inclui nomes completos para inspeção.

In [2]:
def designadores(s):
    tokens=s.split(); final=tokens[-1] if tokens else ''; return tuple(sorted([t for t in tokens if t.isdigit()]+([final] if re.fullmatch('[A-Z]|[IVXLCDM]{2,4}',final) else [])))
nomes=pd.DataFrame([{'original':a,'conciliado':b,'designadores_original':' '.join(designadores(a)), 'designadores_conciliado':' '.join(designadores(b)), 'designador_alterado':designadores(a)!=designadores(b)} for a,b in mapa.items()])
cont=lc0.value_counts(); nomes['registros']=nomes.original.map(cont)
nomes.to_csv(T/'nb27_conciliacoes_designadores.csv',index=False)
suspeitos=set(nomes.loc[nomes.designador_alterado,'original'])
lc_prot=lc0.map(lambda s:mapa.get(s,s) if s not in suspeitos else s)
kp=(lc_prot.astype('string')+'|'+nc.astype('string')).where(nc.notna())
base=kc.isin(set(kb.dropna())); prot=kp.isin(set(kb.dropna()))
vila=gpd.read_parquet(P/'vila_favela.parquet',columns=['geometry']).geometry.union_all()
em_vila=pd.Series(shapely.within(C.geometry.to_numpy(),vila),index=C.index)
lin=[]
for grupo,m in [('todos',pd.Series(True,index=C.index)),('assentamentos',em_vila),('fora',~em_vila)]:
    lin.append({'grupo':grupo,'registros':int(m.sum()),'pares_base':int((base&m).sum()),'pares_designador_preservado':int((prot&m).sum()),'perdas':int((base&~prot&m).sum())})
pd.DataFrame(lin).to_csv(T/'nb27_cobertura_designadores.csv',index=False)
print(nomes[nomes.designador_alterado].to_string(index=False)); print(pd.DataFrame(lin).to_string(index=False))


             original         conciliado designadores_original designadores_conciliado  designador_alterado  registros
            BECO B UM           BEC UM B                                             B                 True         15
         BECO CINCO B        BEC B CINCO                     B                                         True         12
    BECO DA AMIZADE 2     BEC DA AMIZADE                     2                                         True          9
    BECO DAS FLORES 2     BEC DAS FLORES                     2                                         True          9
      BECO ITAPIRANGA   BEC ITAPIRANGA A                                             A                 True         37
    BECO PEDRA AZUL 2     BEC PEDRA AZUL                     2                                         True         40
    BECO PONTA PORA 1     BEC PONTA PORA                     1                                         True         63
BECO TRINTA E CINCO A BEC TRINTA E CINCO        

## 2. Contraste entre apartamentos e demais registros em nível 1

Mesmos registros elegíveis por lote único do NB23, excluindo estabelecimentos. Quintis e pesos calculados no conjunto básico, fixos em ambos os recortes e nas 500 réplicas por blocos de 2 km, localizados pela coordenada do CNEFE. Reamostragem de blocos ocupados com igual probabilidade e reposição. O intervalo é condicional a essa padronização. Nível 1 não garante ausência de erro de coordenada nem de correspondência.

Os intervalos desta sensibilidade usam semente 42 em cada recorte, em sequência diferente da empregada no NB23. Os estimadores pontuais básicos foram reproduzidos.

In [3]:
iptu=pd.read_csv(ROOT/'data/raw/bhmap_context/CADASTRO_IMOBILIARIO.csv',usecols=['NULOTCTM','TIPO_LOGRADOURO','NOME_LOGRADOURO','NUMERO_IMOVEL'],dtype=str)
li0,ni=chave_identidade(iptu.TIPO_LOGRADOURO.fillna('').str.replace(r'^BEC$','BECO',regex=True)+' '+iptu.NOME_LOGRADOURO.fillna(''),iptu.NUMERO_IMOVEL)
mi=conciliar_logradouros(li0.unique(),lb.unique(),limiar=90,margem=2)
li=li0.map(lambda s:mi.get(s,s)); ki=(li.astype('string')+'|'+ni.astype('string')).where(ni.notna())
kl=pd.DataFrame({'chave':ki,'NULOTCTM':iptu.NULOTCTM}).dropna().drop_duplicates()
lotes=gpd.read_parquet(P/'lote_ctm.parquet',columns=['NULOTCTM','AREA_M2','geometry']).drop_duplicates('NULOTCTM').set_index('NULOTCTM')
kl=kl[kl.NULOTCTM.isin(lotes.index)]
unic=kl.groupby('chave').NULOTCTM.nunique(); lookup=kl[kl.chave.isin(unic[unic.eq(1)].index)].set_index('chave').NULOTCTM
d=pd.read_parquet(T/'nb23_ponto_lote_pares.parquet').join(pd.DataFrame({'apto':apto,'estab':estab,'nivel':C.NV_GEO_COORD,'chave':kc,'x':C.geometry.x,'y':C.geometry.y}))
d['lote']=d.chave.map(lookup); d['area']=d.lote.map(lotes.AREA_M2)
d=d[~d.estab & d.area.notna()].copy(); d['q']=pd.qcut(d.area,5,labels=False)
d['bloco']=(d.x//2000).astype(int).astype(str)+'_'+(d.y//2000).astype(int).astype(str)
pesos=np.bincount(d.loc[d.apto,'q'],minlength=5)/d.apto.sum()
def estimate(v,q,a):
    return sum(pesos[k]*(np.median(v[(q==k)&a],axis=0)-np.median(v[(q==k)&~a],axis=0)) for k in range(5))
lin=[]
for nome,s in [('todos_niveis',d),('nivel_1',d[d.nivel.eq(1)])]:
    v=s[['dist','dC']].to_numpy(); q=s.q.to_numpy(); a=s.apto.to_numpy()
    assert all(((q==k)&a).any() and ((q==k)&~a).any() for k in range(5))
    pos=list(s.groupby('bloco').indices.values()); rng=np.random.default_rng(42); reps=[]
    for _ in range(500):
        ix=np.concatenate([pos[k] for k in rng.integers(len(pos),size=len(pos))]); reps.append(estimate(v[ix],q[ix],a[ix]))
    val=estimate(v,q,a); lo,hi=np.percentile(reps,[2.5,97.5],axis=0)
    for j,medida in enumerate(['ponto_ponto','ponto_poligono']):
        lin.append({'recorte':nome,'medida':medida,'registros':len(s),'apartamentos':int(a.sum()),'contraste_m':val[j],'ic95_inf':lo[j],'ic95_sup':hi[j]})
r=pd.DataFrame(lin); r.to_csv(T/'nb27_contraste_nivel1.csv',index=False)
assert np.isclose(r.iloc[0].contraste_m,4.80538778,atol=1e-6,rtol=0)
print(r.to_string(index=False))


     recorte         medida  registros  apartamentos  contraste_m  ic95_inf  ic95_sup
todos_niveis    ponto_ponto     730774        363643     4.805388  3.655956  6.112586
todos_niveis ponto_poligono     730774        363643     0.976683  0.518577  1.464176
     nivel_1    ponto_ponto     569217        210114     6.405821  5.028672  7.879968
     nivel_1 ponto_poligono     569217        210114     1.199353  0.581311  1.825911


## 3. Condições sobrepostas e lotes próximos

Frequências marginais não são classes causais. A condição CNEFE longe do lote ligado próximo ao ponto municipal pode coexistir com a ambiguidade da chave, mas é logicamente incompatível com a condição de proximidade de ambos os pontos aos lotes, com tolerância fixa de 10 m. Recalculamos os identificadores dos lotes mais próximos apenas para decompor a classe de proximidade de ambos os pontos. Nenhum rótulo básico é alterado.


In [4]:
flags=pd.DataFrame({'ambiguidade_bbox':cauda.ambigua,'CNEFE_longe_BHMap_perto':cauda.dC.gt(10)&cauda.dB.le(10),'conciliada':cauda.conciliada,'ambos_perto_lotes':cauda.dC.le(10)&cauda.dB.le(10)})
lin=[]
for a in flags:
    for b in flags:
        lin.append({'condicao_a':a,'condicao_b':b,'pares':int((flags[a]&flags[b]).sum()),'denominador':len(cauda)})
pd.DataFrame(lin).to_csv(T/'nb27_condicoes_sobrepostas.csv',index=False)
pd.crosstab(cauda.veredito,flags.CNEFE_longe_BHMap_perto).to_csv(T/'nb27_fora_lote_por_classe.csv')
s=cauda[cauda.veredito.eq('imovel extenso, mesmo lote')]
e=s[['chave','xc','yc','xb','yb']].reset_index().rename(columns={'index':'id'}).merge(kl,on='chave')
idcol=s.index.name or 'id'
if idcol not in e: idcol=e.columns[0]
g=lotes.geometry.reindex(e.NULOTCTM).to_numpy()
e['dc']=shapely.distance(shapely.points(e.xc.to_numpy(),e.yc.to_numpy()),g); e['db']=shapely.distance(shapely.points(e.xb.to_numpy(),e.yb.to_numpy()),g)
a=e.loc[e.groupby(idcol).dc.idxmin()].set_index(idcol).NULOTCTM
b=e.loc[e.groupby(idcol).db.idxmin()].set_index(idcol).NULOTCTM
counts=pd.Series({'mesmo_lote':int(a.eq(b).sum()),'lotes_diferentes_proximos':int(a.ne(b).sum()),'total_classe':len(s)})
assert counts.mesmo_lote+counts.lotes_diferentes_proximos==9521
counts.to_csv(T/'nb27_decomposicao_lotes.csv',header=['pares'])
print(counts.to_string());print(pd.crosstab(cauda.veredito,flags.CNEFE_longe_BHMap_perto).to_string())


mesmo_lote                   9520
lotes_diferentes_proximos       1
total_classe                 9521
CNEFE_longe_BHMap_perto            False  True 
veredito                                       
BHMap fora do lote fiscal             91      0
CNEFE fora da via nomeada           3754      0
CNEFE fora do lote fiscal              0   7662
grafia conciliada sem confirmacao    914      0
homonimia, chave em dois lugares    4683    832
imovel extenso, mesmo lote          9521      0
indeterminado                        156      0
mesma via sem lote fiscal           2162      0


In [5]:
from pathlib import Path
import hashlib, json
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/config.py').exists())
files=['data/processed/cnefe_bh.parquet','data/processed/bhmap_bh.parquet','data/processed/lote_ctm.parquet','data/processed/vila_favela.parquet','data/raw/bhmap_context/CADASTRO_IMOBILIARIO.csv','outputs/referencia_completa/tables/nb23_cauda_adjudicada.parquet','outputs/referencia_completa/tables/nb23_ponto_lote_pares.parquet']
manifest={n:hashlib.sha256((root/n).read_bytes()).hexdigest() for n in files}
(root/'outputs/referencia_completa/tables/nb27_manifesto.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Hashes registrados:',len(manifest))


Hashes registrados: 7


## 4. Distâncias nos registros recuperados e caudas sob restrição territorial

Separamos os registros com chave disponível no BHMap antes da conciliação daqueles que passam a ter candidato depois dela. Usamos a chave normalizada e as distâncias do pareamento completo produzido no [NB23](23_referencia_municipal_completa.ipynb), mantendo o candidato mais próximo. A comparação é descritiva: os grupos diferem em composição territorial e não constituem um experimento sobre o efeito da conciliação.

Reunimos também as caudas acima de 100 m e de 1 km das restrições do NB23 e do [NB25](25_setor_censitario.ipynb). Cada porcentagem usa como denominador os pares retidos pela respectiva regra. Essa seleção precisa acompanhar a interpretação da redução das caudas.


In [6]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.metrics import chave_identidade, separar_logradouro_numero

tab = config.OUTPUT_DIR / 'referencia_completa/tables'
entradas = [config.PROCESSED_DIR/'cnefe_bh.parquet',
            config.PROCESSED_DIR/'bhmap_bh.parquet',
            config.PROCESSED_DIR/'cnefe_identidade_completa.parquet',
            tab/'nb23_restricao_territorial.csv', tab/'nb25_restricao_setor.csv']
def hash_arquivo(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for bloco in iter(lambda: f.read(1024*1024), b''):
            h.update(bloco)
    return h.hexdigest()
hashes = {p.relative_to(ROOT).as_posix(): hash_arquivo(p) for p in entradas}
c = pd.read_parquet(entradas[0], columns=['LOGRAD_NUM', 'COD_UNICO_ENDERECO'])
b = pd.read_parquet(entradas[1], columns=['SIGLA_TIPO', 'NOME_LOGRA', 'NUMERO_IMO'])
i = pd.read_parquet(entradas[2])
assert c.index.equals(i.index)
assert c.COD_UNICO_ENDERECO.equals(i.COD_UNICO_ENDERECO)
lc, nc = chave_identidade(*separar_logradouro_numero(c.LOGRAD_NUM))
lb, nb = chave_identidade(b.SIGLA_TIPO.fillna('')+' '+b.NOME_LOGRA.fillna(''), b.NUMERO_IMO)
lc.index = nc.index = c.index
lb.index = nb.index = b.index
kc = (lc.astype('string')+'|'+nc.astype('string')).where(nc.notna())
kb = (lb.astype('string')+'|'+nb.astype('string')).where(nb.notna())
exato = kc.isin(set(kb.dropna()))
pareado = i.pareado_identidade.fillna(False).astype(bool)
assert not (exato & ~pareado).any()
recuperado = pareado & ~exato
assert pareado.sum() == 1062044 and recuperado.sum() == 36414
linhas = []
for grupo, mascara in [('exato', exato), ('recuperado', recuperado), ('todos', pareado)]:
    d = i.loc[mascara, 'dist_identidade']
    assert d.notna().all() and d.ge(0).all()
    linhas.append(dict(grupo=grupo, pares=len(d), mediana_m=d.median(), p90_m=d.quantile(.9),
                       acima100_n=int(d.gt(100).sum()), acima100_pct=100*d.gt(100).mean(),
                       acima1000_n=int(d.gt(1000).sum())))
conciliacao = pd.DataFrame(linhas)
assert conciliacao.iloc[:2].pares.sum() == conciliacao.iloc[2].pares
assert conciliacao.iloc[:2].acima100_n.sum() == 29775
conciliacao.to_csv(tab/'nb27_distancias_conciliacao.csv', index=False)
print(conciliacao.to_string(index=False))

r = pd.read_csv(entradas[3])
assert (r.cauda100_original-r.cauda100_excluida+r.cauda100_entrada).equals(r.cauda100_final)
assert (r.cauda1000_original-r.cauda1000_excluida+r.cauda1000_entrada).equals(r.cauda1000_final)
territorial = r[['regra', 'pares', 'cauda100_final', 'cauda1000_final']].copy()
s = pd.read_csv(entradas[4], index_col=0).iloc[:,0]
territorial.loc[len(territorial)] = ['mais_setor', int(s['pares']), int(s['cauda100_final']), int(s['cauda1000_final'])]
territorial['cauda100_pct'] = 100*territorial.cauda100_final/territorial.pares
territorial['cauda1000_pct'] = 100*territorial.cauda1000_final/territorial.pares
territorial.to_csv(tab/'nb27_caudas_restricoes.csv', index=False)
print(territorial.to_string(index=False))
assert all(hash_arquivo(p) == hashes[p.relative_to(ROOT).as_posix()] for p in entradas)
(tab/'nb27_manifesto_conciliacao_caudas.json').write_text(json.dumps(hashes, indent=2), encoding='utf-8')


     grupo   pares  mediana_m     p90_m  acima100_n  acima100_pct  acima1000_n
     exato 1025630   9.796325 35.023087       27056      2.637988         6117
recuperado   36414  11.157738 54.840684        2719      7.466908         1984
     todos 1062044   9.837064 35.589708       29775      2.803556         8101
            regra   pares  cauda100_final  cauda1000_final  cauda100_pct  cauda1000_pct
logradouro_numero 1062044           29775             8101      2.803556       0.762774
      mais_bairro  972070           18010               81      1.852747       0.008333
         mais_cep  971505           21082             1862      2.170035       0.191661
mais_bairro_e_cep  900004           16488               59      1.831992       0.006556
       mais_setor 1038425           15719                5      1.513735       0.000481


Os 36.414 registros recuperados apresentam mediana de 11,16 m, P90 de 54,84 m e 7,47% acima de 100 m. Nos 1.025.630 registros pareados sem conciliação, os valores são 9,80 m, 35,02 m e 2,64%. A pequena mudança da mediana global não descreve, portanto, toda a distribuição do grupo recuperado. Essas diferenças não identificam a causa das discrepâncias nem validam as correspondências.

Com bairro e CEP, restam 900.004 pares, dos quais 16.488 estão acima de 100 m (1,83%) e 59 acima de 1 km. A redução para 59 casos não significa que toda a cauda acima de 100 m desapareceu. O denominador e a composição dos pares também mudam com cada restrição.
